THe CNN model

In [ ]:
import pandas as pd 
import numpy as np 
import os 
from tensorflow.keras.preprocessing.image import ImageDataGenerator,load_img,img_to_array
from tensorflow.keras.models import Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Flatten,Dense,Dropout,GlobalAveragePooling2D

In [ ]:
dataset_path='C:\projects\Pokemon Images DB'
datagen = ImageDataGenerator(
    rotation_range=25,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.3,
    horizontal_flip=True,
    fill_mode='nearest'
)
for folder in os.listdir(dataset_path):
    folder_path = os.path.join(dataset_path, folder)

    if os.path.isdir(folder_path):
        for img_name in os.listdir(folder_path):
            img_path = os.path.join(folder_path, img_name)

            img = load_img(img_path)
            x = img_to_array(img)
            x = np.expand_dims(x, axis=0)

            i = 0
            for batch in datagen.flow(x, batch_size=1,
                                      save_to_dir=folder_path,
                                      save_prefix="aug",
                                      save_format="jpg"):
                i += 1
                if i >= 3:   # only 2 new images per image
                    break

In [ ]:
path='C:\projects\Pokemon Images DB'
image_path=ImageDataGenerator(rescale=1./255,validation_split=0.2)
train_data=image_path.flow_from_directory(path,target_size=(128,128),color_mode="rgb",subset="training",class_mode="categorical")
validation_data=image_path.flow_from_directory(path,target_size=(128,128),color_mode="rgb",subset="validation",class_mode="categorical")
print(len(train_data.class_indices))

In [ ]:
from tensorflow.keras.optimizers import Adam
basic_model=MobileNetV2(weights='imagenet',include_top=False,input_shape=(128,128,3))
for layers in basic_model.layers:
    layers.trainable=False
x=GlobalAveragePooling2D()(basic_model.output)
x=Dense(512,activation='relu',kernel_initializer='uniform')(x)
x=Dropout(0.3)(x)
x=Dense(256,activation='relu',kernel_initializer='uniform')(x)
prediction=Dense(len(train_data.class_indices),activation='softmax',kernel_initializer='uniform')(x)
model=Model(inputs=basic_model.input,outputs=prediction)
model.compile(optimizer = Adam(learning_rate=0.0001),loss='categorical_crossentropy',metrics=['accuracy'])
model.fit(train_data,validation_data=validation_data,epochs=25,batch_size=32,verbose=1)




In [ ]:
for layer in basic_model.layers[-30:]:
    layer.trainable = True

# Recompile with lower learning rate
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Fine-tuning last layers...")

history_fine = model.fit(
    train_data,
    validation_data=validation_data,
    epochs=15
)

model.save("pokemon_model.h5")

In [ ]:
model.save("pokemon_model.h5")

In [ ]:
loss,acc=model.evaluate(validation_data)
print(acc)

In [ ]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import numpy as np
from tensorflow.keras.models import load_model
from PIL import Image

path='C:\projects\Pokemon Images DB'
image_path=ImageDataGenerator(rescale=1./255,validation_split=0.2)
train_data=image_path.flow_from_directory(path,target_size=(128,128),color_mode="rgb",subset="training",class_mode="categorical")

mod=load_model("pokedex_model.h5")
# Path to a single image you want to predict

# Step 1: Load image with PIL
import numpy as np
from PIL import Image
from tensorflow.keras.preprocessing.image import img_to_array

single_image_path = r"C:\projects\deeplnproject\image.png"

# Step 1: Load image in RGB (3 channels)
img = Image.open(single_image_path).convert("RGB")

# Step 2: Create a white background (same size)
white_bg = Image.new("RGB", img.size, (255, 255, 255))

# Step 3: Paste the image onto the white background
white_bg.paste(img, (0, 0))

# Step 4: Resize to target size
final_img = white_bg.resize((128, 128))

# Step 5: Convert to array
img_array = img_to_array(final_img)

# Step 6: Normalize
img_array = img_array / 255.0

# Step 7: Add batch dimension
img_array = np.expand_dims(img_array, axis=0)

# Step 8: Predict
prediction = mod.predict(img_array)
print("Prediction:", prediction)

# Get class indices mapping
class_indices = train_data.class_indices
labels = dict((v, k) for k, v in class_indices.items())   # reverse mapping

# Find predicted class
predicted_class = labels[pr:=np.argmax(prediction)]
print("Predicted class:", predicted_class)